## Merge Logic

This code creates a final meme template label by combining:

- the original `template`
- the manually reviewed annotations
- the automatic text/image template predictions

### Priority Order

1. **Keep the original template**
   - If `template` is already not `NO_TEMPLATE`, leave it unchanged.

2. **Use manual review if available**
   - If the row was reviewed and the original `template` is `NO_TEMPLATE`:
     - if both text and image were accepted, use the **image** template
     - if only image was accepted, use the **image** template
     - if only text was accepted, use the **text** template
     - if both were rejected, use `NO_TEMPLATE`

3. **Use automatic fallback if not reviewed**
   - If the row was not reviewed and the original `template` is `NO_TEMPLATE`:
     - if both `img_template` and `text_template` exist, use **image**
     - if only `img_template` exists, use **image**
     - if only `text_template` exists, use **text**
     - otherwise keep `NO_TEMPLATE`

### Final Result
The final label is stored in `template_final`.


In [8]:
import numpy as np
import pandas as pd

MAIN_PATH = "data/new_assigned_memes.csv"
ANN_PATH = "data/template_label_sheet_annotated.csv"
OUT_PATH = "data/new_assigned_memes_final_merged.csv"

NO = "NO_TEMPLATE"
KEY = "key"

# Load
main_df = pd.read_csv(MAIN_PATH, low_memory=False)
ann_df = pd.read_csv(ANN_PATH, low_memory=False)

# Normalize keys
main_df[KEY] = main_df[KEY].astype(str).str.strip()
ann_df[KEY] = ann_df[KEY].astype(str).str.strip()

# Keep only annotation columns needed for review logic
ann_keep = ann_df[
    [KEY, "text_template", "img_template", "is_correct_text", "is_correct_img"]
].drop_duplicates(subset=[KEY]).copy()

# Normalize review flags
ann_keep["is_correct_text"] = pd.to_numeric(ann_keep["is_correct_text"], errors="coerce")
ann_keep["is_correct_img"] = pd.to_numeric(ann_keep["is_correct_img"], errors="coerce")

# Normalize template strings inside annotation file
for col in ["text_template", "img_template"]:
    ann_keep[col] = ann_keep[col].fillna(NO).astype(str).str.strip()
    ann_keep.loc[ann_keep[col].eq(""), col] = NO

# Merge annotation data onto main dataset
df = main_df.merge(
    ann_keep.rename(
        columns={
            "text_template": "ann_text_template",
            "img_template": "ann_img_template",
        }
    ),
    on=KEY,
    how="left",
)

# Normalize template strings in main dataset for fallback logic
for col in ["template", "text_template", "img_template"]:
    df[col] = df[col].fillna(NO).astype(str).str.strip()
    df.loc[df[col].eq(""), col] = NO

# A row is "reviewed" only if at least one correctness flag is filled in
df["was_reviewed"] = df["is_correct_text"].notna() | df["is_correct_img"].notna()

# Per-method reviewed outputs
df["text_reassigned_tpl"] = np.where(
    df["is_correct_text"].eq(1),
    df["ann_text_template"],
    NO,
)

df["img_reassigned_tpl"] = np.where(
    df["is_correct_img"].eq(1),
    df["ann_img_template"],
    NO,
)

# Manual decision from reviewed rows
def choose_reviewed_template(row):
    text_ok = row["is_correct_text"] == 1
    img_ok = row["is_correct_img"] == 1

    if text_ok and img_ok:
        # If both accepted, prefer image
        return row["ann_img_template"] if row["ann_img_template"] != NO else row["ann_text_template"]

    if img_ok:
        return row["ann_img_template"]

    if text_ok:
        return row["ann_text_template"]

    # both 0, or unreviewed/nulls
    return NO

df["reviewed_template"] = df.apply(choose_reviewed_template, axis=1)

# Auto fallback from main dataset for rows not reviewed
# Rule: if both have templates, prefer image
text_has = df["text_template"].ne(NO)
img_has = df["img_template"].ne(NO)

df["auto_fallback_template"] = np.select(
    [
        img_has & text_has,
        img_has & ~text_has,
        ~img_has & text_has,
    ],
    [
        df["img_template"],
        df["img_template"],
        df["text_template"],
    ],
    default=NO,
)

# Final priority:
# 1. Keep original template if already labeled
# 2. If original is NO_TEMPLATE and row was reviewed, use reviewed decision
# 3. If original is NO_TEMPLATE and not reviewed, use auto fallback
# 4. Else NO_TEMPLATE
df["template_final"] = df["template"]

orig_no_mask = df["template"].eq(NO)
reviewed_no_mask = orig_no_mask & df["was_reviewed"]
unreviewed_no_mask = orig_no_mask & ~df["was_reviewed"]

df.loc[reviewed_no_mask, "template_final"] = df.loc[reviewed_no_mask, "reviewed_template"]
df.loc[unreviewed_no_mask, "template_final"] = df.loc[unreviewed_no_mask, "auto_fallback_template"]

# Save
df.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}")

# Quick summary
print("Total rows:", len(df))
print("Originally labeled (not NO_TEMPLATE):", df["template"].ne(NO).sum())
print("Reviewed rows:", df["was_reviewed"].sum())
print("Original NO_TEMPLATE:", df["template"].eq(NO).sum())
print("Final NO_TEMPLATE:", df["template_final"].eq(NO).sum())
print(
    "Recovered from NO_TEMPLATE:",
    ((df["template"].eq(NO)) & (df["template_final"].ne(NO))).sum()
)

# Optional breakdown
summary = pd.DataFrame({
    "count": [
        df["template"].eq(NO).sum(),
        df["template"].ne(NO).sum(),
        df["template_final"].eq(NO).sum(),
        df["template_final"].ne(NO).sum(),
    ]
}, index=[
    "original_no_template",
    "original_has_template",
    "final_no_template",
    "final_has_template",
])
print(summary)


Saved: data/new_assigned_memes_final_merged.csv
Total rows: 171793
Originally labeled (not NO_TEMPLATE): 79835
Reviewed rows: 265
Original NO_TEMPLATE: 91958
Final NO_TEMPLATE: 78202
Recovered from NO_TEMPLATE: 13756
                       count
original_no_template   91958
original_has_template  79835
final_no_template      78202
final_has_template     93591


In [9]:
total = len(df)
orig_no = df["template"].eq(NO).sum()
final_no = df["template_final"].eq(NO).sum()

print(f"Original NO_TEMPLATE: {orig_no} ({orig_no/total:.2%})")
print(f"Final NO_TEMPLATE: {final_no} ({final_no/total:.2%})")
print(f"Reduction: {orig_no - final_no} rows ({(orig_no - final_no)/total:.2%} of dataset)")


Original NO_TEMPLATE: 91958 (53.53%)
Final NO_TEMPLATE: 78202 (45.52%)
Reduction: 13756 rows (8.01% of dataset)
